# 03 教師なし手法 — 「期待値モデル」を作らずに異常を見つける

## ゴール
- **Isolation Forest** と **LOF** という教師なし手法で、異常を検知する。
- 02 の残差法（天気から期待値を予測する）と比べて、**何が違い、何が同じか**を確かめる。
- **特徴量の作り方**（人がどこまで教えるか）で、結果が大きく変わることを理解する。

## 前のNotebookとのつながり
02 では、「日射量から発電量を予測する直線」という**期待値モデル**を、人が設計しました。
ここでは、そのようなモデルを作らず、**正常データの分布から外れた日**を異常とする手法を使います。

## やること
1. 3通りの特徴量（A: 素の値 / B: ＋比 / C: ＋時間的な特徴）を見る
2. 2手法 × 3特徴量を、02 と同じ条件（10系列・しきい値は正常日の警報率3%）で評価する
3. 「異常のない系列」で、しきい値が狙いどおりに働くかを確かめる

## キーワード
- **Isolation Forest**: データをランダムに分割していき、**少ない分割で孤立する点ほど異常**とみなす手法。
- **LOF（局所外れ値因子）**: ある点の周りの密度を、近くの点の密度と比べ、**周りより疎な点ほど異常**とみなす手法。
- **特徴量**: 手法に渡す数値の列。何を渡すかが、結果を大きく左右する。
- **特徴量設計**: 異常を見つけやすい特徴量を、人が工夫して作ること。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from anomaly_synth import make_split
from anomaly_eval import alarm_by_normal_quantile
from baselines import DETECTOR_COLUMNS
from unsup import FEATURE_SETS, UnsupDetector, make_features, out_of_fold_score
import run_phase3

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
train = make_split("train")[DETECTOR_COLUMNS]
val = make_split("val", 1)
x = val[DETECTOR_COLUMNS]

for f, desc in FEATURE_SETS.items():
    print(f, ":", desc, "→ 列:", make_features(x, f).columns.tolist())
make_features(x, "C").head(3).round(2)

**何をしているか**: 3通りの特徴量セットと、その列を表示します。

**出力の見方**:
- **A**は、観測値をそのまま渡します。「発電量が日射量に比例する」という関係は、**手法が自力で見つける**必要があります。
- **B**は、**発電量÷日射量**（比）を足します。02 で見た「期待値からのずれ」とほぼ同じ情報を、人が与えることになります。
- **C**は、さらに**比の7日平均**と**前日との差の絶対値**を足します。前者は徐々に進む劣化、後者は値が動かない固着を見つけるための工夫です。
- どの特徴量にも、答え（`is_anomaly` など）は含まれていません。

In [ ]:
det = UnsupDetector("iforest", "A").fit(train)
thr = alarm_by_normal_quantile(out_of_fold_score(train, "iforest", "A"), 0.03)
score = det.score(x)
alarm = (score > thr).to_numpy()
normal = (val["is_anomaly"] == 0).to_numpy()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
ax[0].scatter(x["gsr"][normal & ~alarm], x["gen_kwh"][normal & ~alarm], s=8, color="lightgray", label="正常日（警報なし）")
ax[0].scatter(x["gsr"][normal & alarm], x["gen_kwh"][normal & alarm], s=25, color="blue", label="正常日（誤警報）")
ax[0].scatter(x["gsr"][~normal & alarm], x["gen_kwh"][~normal & alarm], s=25, color="red", label="異常日（検知）")
ax[0].scatter(x["gsr"][~normal & ~alarm], x["gen_kwh"][~normal & ~alarm], s=25, color="orange", marker="x", label="異常日（見逃し）")
ax[0].set(title="Isolation Forest・特徴A の警報", xlabel="日射量 gsr", ylabel="発電量 kWh"); ax[0].legend(fontsize=7)

det = UnsupDetector("iforest", "B").fit(train)
thr = alarm_by_normal_quantile(out_of_fold_score(train, "iforest", "B"), 0.03)
alarm = (det.score(x) > thr).to_numpy()
ax[1].scatter(x["gsr"][normal & ~alarm], x["gen_kwh"][normal & ~alarm], s=8, color="lightgray")
ax[1].scatter(x["gsr"][normal & alarm], x["gen_kwh"][normal & alarm], s=25, color="blue")
ax[1].scatter(x["gsr"][~normal & alarm], x["gen_kwh"][~normal & alarm], s=25, color="red")
ax[1].scatter(x["gsr"][~normal & ~alarm], x["gen_kwh"][~normal & ~alarm], s=25, color="orange", marker="x")
ax[1].set(title="Isolation Forest・特徴B の警報", xlabel="日射量 gsr")
fig.tight_layout()

**何をしているか**: 発電量と日射量の散布図に、Isolation Forest の警報を重ねます（左が特徴A、右が特徴B。同じ val の1系列）。

**出力の見方**:
- 左（特徴A）: 青い点（誤警報）と赤い点（検知）は、散布図の**端**（日射量や発電量が極端に大きい・小さい日）に集まります。
  Isolation Forest は「**値が珍しい**日」を異常とみなすので、**「晴天の夏の日」のように珍しいだけの正常日**も警報にしてしまいます。
  一方、**「値は普通だが、日射量の割に発電が少ない」日**（影・劣化）は、珍しくないので見逃します（×印）。
- 右（特徴B）: 比（発電量÷日射量）を渡すと、**発電量と日射量の関係から外れた日**を見つけられます。
- 「異常」とは「珍しい」ことではなく、「**天気の割におかしい**」ことです。特徴量でそれを教えないと、手法は「珍しさ」しか見ません。

In [ ]:
res, agg, chance = run_phase3.run(verbose=False)
agg

**何をしているか**: 2手法 × 3特徴量を、02 と同じ条件（val の10系列、しきい値は学習に使っていない年のスコアの上位3%）で評価します。最下行は、比較のための 02 の残差法です。

**出力の見方**:
- **特徴Aは弱い**。Isolation Forest の `auprc` は0.275、`precision` は0.327です。警報の3分の2が誤りで、`auprc` は天気を考えない 01 の方法（0.32〜0.35）と同程度かそれ以下です。
- **特徴B（比）を足すと大きく改善**します。IFの`auprc`は0.275→0.530、LOFは0.438→0.649です。LOF・特徴B は 02 の残差法（0.687）に近くなります（標準偏差0.06前後の範囲内）。
- **特徴C（時間的な特徴）では、誤警報が増えて見えます**（IF 2.47、LOF 4.52）。`fa_grace7`（異常直後7日の正常日を除いた誤警報）では、0.69と0.55に下がるので、**7日平均による余韻**が原因です（02 と同じ）。
- 結局、**手法の違い（IFかLOFか）よりも、特徴量の違い（AかBか）の方がずっと大きな差**を生みます。教師なし手法でも、人が「天気に対する比を見る」ことを教えているのです。

In [ ]:
print("== 検知率（イベント単位） ==")
print(res.pivot(index="detector", columns="type", values="recall").loc[agg.index])
print("\n== 日単位の検知率 ==")
print(res.pivot(index="detector", columns="type", values="day_recall").loc[agg.index])
print("\n偶然の水準:", chance.to_dict())

**出力の見方**:
- **停電**: 特徴B・Cなら全手法で、日単位1.0。特徴Aでは、IFは0.22と大きく見逃します。停電は発電が0に近く、「珍しい」はずですが、**冬の雨の日**も発電が小さく、見分けがつかないためです。
- **影**: LOF・特徴Bは、イベント単位も日単位も1.0です。IF・特徴Bは日単位0.45に下がります（LOFの方が、近くの点との密度の違いに敏感です）。
- **劣化**: 日単位は、LOF・特徴B/C と IF・特徴C で0.20〜0.26、残差法（0.29）と同程度かやや低く、IF・特徴Bは0.04です。**徐々に進む異常は、どの手法でも検知が遅れます**（検知までの日数は、特徴Bで11〜19日）。
- **固着**: LOF・特徴B/C は、イベント単位0.95〜1.0、日単位0.77〜0.82と、**残差法（0.55／0.36）より良い**結果です。
  固着の日は、天気が変わるのに発電量が変わらず、「発電量・日射量・日照率・天気」の組み合わせが、正常データに無いパターンになるためと考えられます。特徴Cの「前日との差」も効いている可能性がありますが、特徴Bだけでも見つかっています。
- ただし、イベント単位は偶然の水準（劣化0.64・影0.14・固着0.13）と比べる必要があります。劣化は、偶然に近い値が多いので注意してください。

In [ ]:
rows = []
for kind in ("iforest", "lof"):
    for f in "ABC":
        det = UnsupDetector(kind, f).fit(train)
        thr = alarm_by_normal_quantile(out_of_fold_score(train, kind, f), 0.03)
        rates = [(det.score(make_split("val", i, events={})[DETECTOR_COLUMNS]) > thr).mean() for i in range(1, 11)]
        rows.append(dict(手法=kind, 特徴量=f, 狙い=0.03, 異常のない系列の警報率=round(float(np.mean(rates)), 3)))
pd.DataFrame(rows)

**何をしているか**: 異常を1つも含まない系列（`events={}`）で、正常日に警報が出る割合を測ります。しきい値は「3%」を狙って決めています。

**出力の見方**:
- Isolation Forest は、狙い（0.03）に近い0.024〜0.028です。
- LOF の特徴Aは0.040、**特徴Bは0.052**と、狙いより多く警報が出ます。しきい値は「年ごとのhold-out」（1年分だけで学習したモデルのスコア）で決めていますが、最終モデルは2年分で学習します。
  学習データの量が変わると、LOFのスコアの分布も変わるためと考えられます。**LOFは、学習データの量に敏感**です。
- LOF・特徴Bの誤警報が月1.5回と、狙い（約0.9回）より多かったのは、このためです。
- 02 の残差法は、このずれが小さいので、**しきい値の決め方の安定性**でも残差法が優れています。

## 試してみよう（演習）
1. **特徴Bだけで、比の代わりに差を使う**: `src/unsup.py` の `make_features` で、`x["ratio"] = ...` を `x["ratio"] = df["gen_kwh"] - 0.85 * df["gsr"]` に変えて、`python src/run_phase3.py` を実行してください。
   見るポイントは、`auprc` の変化です。分かるのは、比と差のどちらが、発電のばらつき（大きさに比例）に合っているかです。
2. **LOFの近傍数を変える**: `src/unsup.py` の `n_neighbors=20` を `5` と `100` に変えて、LOF・特徴Bの結果を比べてください。見るポイントは、影と固着の検知率、異常のない系列の警報率です。分かるのは、近傍数が小さいと局所的な変化に敏感になる一方、不安定になることです。
3. **しきい値を2年分の学習で決める**: 最後のセルで、`out_of_fold_score(train, kind, f)` を `det.score(train)` に置き換えて、警報率がどうなるか比べてください。分かるのは、学習に使ったデータのスコアでしきい値を決めると、LOFで狙いからのずれがどう変わるかです。

## まとめ
- 教師なし手法でも、**特徴量で「天気に対する比」を教えると**、残差法に近い性能になる。教えないと、「珍しいだけの正常日」を誤警報にし、「天気の割におかしい日」を見逃す。
- **手法の差（IF vs LOF）より特徴量の差が大きい**。固着は、LOF の方が残差法より見つけやすかった。
- LOF は学習データの量でスコアの分布が変わり、**しきい値が狙いどおりに働かない**ことがある。残差法のほうが安定していた。
- 7日平均を含む特徴（C）は、02 と同様に**余韻**で誤警報が増えて見える。

次の Phase4 では、**オートエンコーダ**（正常データを圧縮して復元し、うまく復元できない日を異常とみなす）を試します。なお、PyTorch が必要なので、導入の前に確認します。